<a href="https://colab.research.google.com/github/coder-pritom/Conversation-Between-Two-People---LSTM/blob/main/LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Necessary Libraries

In [1]:
import pandas as pd
import numpy as np
import torch
import re
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.optim as optim

# Data Load & Preprocessing

In [2]:
df = pd.read_csv('/content/two_people_conversation_1000.csv')

In [3]:
df.head()

,conversation_id,turn,person_1,person_2,speaker,message,reply_speaker,reply
0,1,1,Alex,Jamie,Alex,"Hey, how was your day?",Jamie,Pretty good! I finished my work early. How abo...
1,2,1,Sam,Taylor,Sam,Morning! Did you sleep well?,Taylor,"I did, thanks. I finally went to bed early."
2,3,1,Riley,Morgan,Riley,Have you been busy this week?,Morgan,"A little, but I'm trying to keep a good balance."
3,4,1,Jordan,Casey,Jordan,Did you finish the grocery shopping?,Casey,Almost. I still need to pick up some fruit.
4,5,1,Alex,Jamie,Alex,Did you get a chance to eat lunch?,Jamie,"Yes, I had rice and vegetables. What did you h..."


In [4]:
df = df[['message','reply']]

In [5]:
df ['token'] = df['message'].str.lower().str.findall(r'\w+') + df['reply'].str.lower().str.findall(r'\w+')

In [6]:
df['token'].iloc[0]

['hey',
 'how',
 'was',
 'your',
 'day',
 'pretty',
 'good',
 'i',
 'finished',
 'my',
 'work',
 'early',
 'how',
 'about',
 'yours']

# Vocab Build & Token IDs

In [7]:
vocab = {'<UNK>':0}

In [8]:
def build_vocab(tokens):
  for token in tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [9]:
_ = df['token'].apply(build_vocab)

In [10]:
vocab

{'<UNK>': 0,
 'hey': 1,
 'how': 2,
 'was': 3,
 'your': 4,
 'day': 5,
 'pretty': 6,
 'good': 7,
 'i': 8,
 'finished': 9,
 'my': 10,
 'work': 11,
 'early': 12,
 'about': 13,
 'yours': 14,
 'morning': 15,
 'did': 16,
 'you': 17,
 'sleep': 18,
 'well': 19,
 'thanks': 20,
 'finally': 21,
 'went': 22,
 'to': 23,
 'bed': 24,
 'have': 25,
 'been': 26,
 'busy': 27,
 'this': 28,
 'week': 29,
 'a': 30,
 'little': 31,
 'but': 32,
 'm': 33,
 'trying': 34,
 'keep': 35,
 'balance': 36,
 'finish': 37,
 'the': 38,
 'grocery': 39,
 'shopping': 40,
 'almost': 41,
 'still': 42,
 'need': 43,
 'pick': 44,
 'up': 45,
 'some': 46,
 'fruit': 47,
 'get': 48,
 'chance': 49,
 'eat': 50,
 'lunch': 51,
 'yes': 52,
 'had': 53,
 'rice': 54,
 'and': 55,
 'vegetables': 56,
 'what': 57,
 'music': 58,
 'do': 59,
 'listen': 60,
 'lately': 61,
 'mostly': 62,
 'indie': 63,
 'acoustic': 64,
 'songs': 65,
 'should': 66,
 'we': 67,
 'cook': 68,
 'tonight': 69,
 'vegetable': 70,
 'fried': 71,
 'it': 72,
 's': 73,
 'quick': 74,


In [11]:
def build_token_ids(tokens):
  token_ids = []
  for token in tokens:
    if token in vocab:
      token_ids.append(vocab[token])
    else:
      token_ids.append(vocab['<UNK>'])
  return token_ids

In [12]:
df['token_id'] = df['token'].apply(build_token_ids)

# Separates Time Steps & Padding

In [13]:
input_data = []
def separate_time_steps(token_id):
  for i in range(2,len(token_id)):
    input_data.append(token_id[:i])

In [14]:
_ = df['token_id'].apply(separate_time_steps)

In [15]:
max_input_length = max(map(len,input_data))

In [16]:
max_input_length

25

In [17]:
def add_padding(input):
  return [0]*(max_input_length - len(input)) + input

input_data = list(map(add_padding,input_data))

# Tensor - Input - Target - Dataset - Dataloader

In [18]:
input_data = torch.tensor(input_data)

In [19]:
X = input_data[:,:-1]
y = input_data[:,-1]

In [20]:
X

tensor([[  0,   0,   0,  ...,   0,   0,   1],
        [  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        ...,
        [  0,   0,   0,  ..., 302, 303,  93],
        [  0,   0,   0,  ..., 303,  93,  30],
        [  0,   0,   0,  ...,  93,  30, 304]])

In [21]:
y

tensor([  2,   3,   4,  ...,  30, 304,   7])

In [22]:
class MyDataset(Dataset):
  def __init__(self,X,y):
    super().__init__()

    self.sequence = X
    self.next = y

  def __len__(self):
    return len(self.sequence)

  def __getitem__(self,idx):
    return self.sequence[idx],self.next[idx]

dataset = MyDataset(X,y)

In [23]:
dataset[0]

(tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]),
 tensor(2))

In [24]:
dataloader = DataLoader(dataset,batch_size=32,shuffle=True)

In [25]:
for sequence,next in dataloader:
  print(sequence.shape,next.shape)
  break

torch.Size([32, 24]) torch.Size([32])


# Model - Epochs - Criterion - Optimizer - Training

In [26]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [27]:
class MyLSTM(nn.Module):
  def __init__(self,num_words,embadding_dim,hidden_size):
    super().__init__()

    self.embadding = nn.Embedding(num_words,embadding_dim)
    self.lstm = nn.LSTM(embadding_dim,hidden_size,batch_first=True)
    self.fc = nn.Linear(hidden_size,num_words)

  def forward(self,inputs):
    embadded_inputs = self.embadding(inputs)
    all_hs ,(hs,cs) = self.lstm(embadded_inputs)
    return self.fc(hs.squeeze(0))

model = MyLSTM(num_words = len(vocab), embadding_dim= 100, hidden_size= 150)
model = model.to(device)

In [28]:
epochs = 15
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr = learning_rate)

In [29]:
model.train()
for epoch in range(epochs):
  total_loss = 0.0
  for sequence,next in dataloader:
    optimizer.zero_grad()
    sequence,next = sequence.to(device),next.to(device)
    logits = model(sequence)
    loss = criterion(logits,next)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()* next.size(0)

  print(f'Epoch : {epoch + 1}  | loss: {total_loss/len(dataloader.dataset):.4f}')

Epoch : 1  | loss: 2.3346
Epoch : 2  | loss: 0.3090
Epoch : 3  | loss: 0.2025
Epoch : 4  | loss: 0.1814
Epoch : 5  | loss: 0.1708
Epoch : 6  | loss: 0.1661
Epoch : 7  | loss: 0.1635
Epoch : 8  | loss: 0.1607
Epoch : 9  | loss: 0.1595
Epoch : 10  | loss: 0.1582
Epoch : 11  | loss: 0.1572
Epoch : 12  | loss: 0.1577
Epoch : 13  | loss: 0.1564
Epoch : 14  | loss: 0.1555
Epoch : 15  | loss: 0.1559


In [58]:
vocab_inverse = {num:word for word,num in vocab.items()}

In [64]:
vocab_inverse

{0: '<UNK>',
 1: 'hey',
 2: 'how',
 3: 'was',
 4: 'your',
 5: 'day',
 6: 'pretty',
 7: 'good',
 8: 'i',
 9: 'finished',
 10: 'my',
 11: 'work',
 12: 'early',
 13: 'about',
 14: 'yours',
 15: 'morning',
 16: 'did',
 17: 'you',
 18: 'sleep',
 19: 'well',
 20: 'thanks',
 21: 'finally',
 22: 'went',
 23: 'to',
 24: 'bed',
 25: 'have',
 26: 'been',
 27: 'busy',
 28: 'this',
 29: 'week',
 30: 'a',
 31: 'little',
 32: 'but',
 33: 'm',
 34: 'trying',
 35: 'keep',
 36: 'balance',
 37: 'finish',
 38: 'the',
 39: 'grocery',
 40: 'shopping',
 41: 'almost',
 42: 'still',
 43: 'need',
 44: 'pick',
 45: 'up',
 46: 'some',
 47: 'fruit',
 48: 'get',
 49: 'chance',
 50: 'eat',
 51: 'lunch',
 52: 'yes',
 53: 'had',
 54: 'rice',
 55: 'and',
 56: 'vegetables',
 57: 'what',
 58: 'music',
 59: 'do',
 60: 'listen',
 61: 'lately',
 62: 'mostly',
 63: 'indie',
 64: 'acoustic',
 65: 'songs',
 66: 'should',
 67: 'we',
 68: 'cook',
 69: 'tonight',
 70: 'vegetable',
 71: 'fried',
 72: 'it',
 73: 's',
 74: 'quick',


# Evaluation

In [111]:
def prediction(message,index):
  tokens = re.findall(r'\w+',message.lower())
  if len(tokens) >= len(df['token'].iloc[index]):
    return message
  token_ids = build_token_ids(tokens)
  input = add_padding(token_ids)[1:]
  input = torch.tensor(input,device=device)
  with torch.no_grad():
    model.eval()
    logits = model(input)
    _ , pred = torch.max(logits,dim=0)
    pred = vocab_inverse[pred.item()]
    message = message + " " + pred
    return prediction(message,index)


In [118]:
idx = 0
prediction(df['message'].iloc[idx],idx)

'Hey, how was your day? pretty good i finished my work early how about yours'